In [1]:
suppressPackageStartupMessages({
    library(jsonlite)
    library(tidyverse)
})

# cell types to fine types

In [ ]:
cohort = 'EDP1-EDP2-ARB'
xen_basepath = file.path('/data/srlab/AMP_collab/data/early_disease_synovium/xenium/combined', cohort)

params = data.frame()
for (lineage in c('B_plasma', 'Endothelial', 'Myeloid', 'Stromal', 'T_NK')) {
    cts <- list.dirs(file.path(xen_basepath, lineage), full.names = TRUE, recursive = FALSE)
    cts <- cts[basename(cts) != 'Untyped']
    for (ct_path in cts) {
        chunks = list.files(ct_path, full.names = TRUE, pattern = "*labeltransfer.rds")
        lineage_params = data.frame(
            ct = basename(ct_path), 
            xen_path = chunks, 
            cohort = cohort, 
            cca_path = ifelse(basename(ct_path) %in% c('T', 'Plasma'), # removing B for now as gene expression correlations looked bad
                              paste0('/data/srlab/AMP_collab/lakshay-yakir/6.fine_types/out_ccaweights/', basename(ct_path), '_ccaweights.RDS'), 
                              ""),
            batch_vars = 'cohort'
            )
        params <- dplyr::bind_rows(params, lineage_params)
    }
}

lines <- lapply(seq_len(nrow(params)), function(i) {
    toJSON(as.list(params[i, ]), auto_unbox = TRUE)
})
outpath = file.path(
    "/data/srlab/AMP_collab/lakshay-yakir/6.fine_types/", 
    paste0("2.celltypes_to_finetypes_", cohort, "_params.jsonl")
    )
writeLines(unlist(lines), outpath)